<a href="https://colab.research.google.com/github/gustavocostamiguel/Python_IA/blob/main/DecisionTree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [79]:
#Sempre ir em executar  tudo
# Importa a biblioteca pandas para manipulação de dados em DataFrames.
import pandas as pd
# Importa a biblioteca numpy para operações numéricas, especialmente com arrays.
import numpy as np
# Importa train_test_split para dividir os dados em conjuntos de treino e teste.
from sklearn.model_selection import train_test_split
# Importa GaussianNB, o classificador Naive Bayes Gaussiano.
from sklearn.tree import DecisionTreeClassifier
# Importa LabelEncoder para codificar rótulos categóricos em numéricos.
from sklearn.preprocessing import LabelEncoder
# Importa métricas de avaliação do modelo, como acurácia, precisão, recall, f1-score e relatório de classificação.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
# Importa ConfusionMatrix para visualização da matriz de confusão.
from sklearn.tree import export_graphviz
import graphviz

In [80]:
base = pd.read_csv("insurance.csv",keep_default_na=False)
# Exibe as primeiras 5 linhas do DataFrame para uma visualização inicial dos dados.
base.head()

,Unnamed: 0,GoodStudent,Age,SocioEcon,RiskAversion,VehicleYear,ThisCarDam,RuggedAuto,Accident,MakeModel,...,HomeBase,AntiTheft,PropCost,OtherCarCost,OtherCar,MedCost,Cushioning,Airbag,ILiCost,DrivHist
0,1,False,Adult,Prole,Adventurous,Older,Moderate,EggShell,Mild,Economy,...,City,False,TenThou,Thousand,True,Thousand,Poor,False,Thousand,Many
1,2,False,Senior,Prole,Cautious,Current,None,Football,None,Economy,...,City,True,Thousand,Thousand,True,Thousand,Good,True,Thousand,Zero
2,3,False,Senior,UpperMiddle,Psychopath,Current,None,Football,None,FamilySedan,...,City,False,Thousand,Thousand,False,Thousand,Good,True,Thousand,One
3,4,False,Adolescent,Middle,Normal,Older,None,EggShell,None,Economy,...,Suburb,False,Thousand,Thousand,True,Thousand,Fair,False,Thousand,Zero
4,5,False,Adolescent,Prole,Normal,Older,Moderate,Football,Moderate,Economy,...,City,False,TenThou,Thousand,False,Thousand,Fair,False,Thousand,Many


In [81]:
# Remove a coluna 'Unnamed: 0' do DataFrame 'base', pois geralmente é um índice extra indesejado do CSV.
base = base.drop(columns=['Unnamed: 0'])
# Exibe as primeiras 5 linhas do DataFrame novamente para verificar se a coluna foi removida com sucesso.
base.head()

,GoodStudent,Age,SocioEcon,RiskAversion,VehicleYear,ThisCarDam,RuggedAuto,Accident,MakeModel,DrivQuality,...,HomeBase,AntiTheft,PropCost,OtherCarCost,OtherCar,MedCost,Cushioning,Airbag,ILiCost,DrivHist
0,False,Adult,Prole,Adventurous,Older,Moderate,EggShell,Mild,Economy,Poor,...,City,False,TenThou,Thousand,True,Thousand,Poor,False,Thousand,Many
1,False,Senior,Prole,Cautious,Current,None,Football,None,Economy,Normal,...,City,True,Thousand,Thousand,True,Thousand,Good,True,Thousand,Zero
2,False,Senior,UpperMiddle,Psychopath,Current,None,Football,None,FamilySedan,Excellent,...,City,False,Thousand,Thousand,False,Thousand,Good,True,Thousand,One
3,False,Adolescent,Middle,Normal,Older,None,EggShell,None,Economy,Normal,...,Suburb,False,Thousand,Thousand,True,Thousand,Fair,False,Thousand,Zero
4,False,Adolescent,Prole,Normal,Older,Moderate,Football,Moderate,Economy,Poor,...,City,False,TenThou,Thousand,False,Thousand,Fair,False,Thousand,Many


In [82]:
# Seleciona a 8ª coluna (índice 7) do DataFrame 'base' como a variável alvo (y) e converte para array numpy.
y = base.iloc[:,7].values
# Remove a 8ª coluna (índice 7) do DataFrame 'base' e seleciona as colunas restantes como as variáveis preditoras (x), convertendo para array numpy.
X = base.drop(base.columns[7], axis=1).values
# Exibe as primeiras linhas do array 'x' para verificar os dados das variáveis preditoras.
X

array([[False, 'Adult', 'Prole', ..., False, 'Thousand', 'Many'],
       [False, 'Senior', 'Prole', ..., True, 'Thousand', 'Zero'],
       [False, 'Senior', 'UpperMiddle', ..., True, 'Thousand', 'One'],
       ...,
       [False, 'Senior', 'UpperMiddle', ..., True, 'Thousand', 'Zero'],
       [False, 'Adult', 'Middle', ..., True, 'Thousand', 'Zero'],
       [False, 'Adult', 'Middle', ..., True, 'Thousand', 'Zero']],
      dtype=object)

In [83]:
# Inicializa o LabelEncoder, que será usado para converter rótulos categóricos em números.
labelencoder = LabelEncoder()
# Itera sobre todas as colunas do DataFrame 'X'.
for i in range(X.shape[1]):
  # Verifica se o tipo de dado da coluna atual é 'object' (indicando uma coluna categórica).
  if X[:,i].dtype == "object":
     # Aplica o LabelEncoder para transformar os valores categóricos em números inteiros.
     X[:,i] = labelencoder.fit_transform(X[:,i])

In [84]:
# Divide os dados em conjuntos de treinamento e teste.
# X_treinamento e Y_treinamento serão usados para treinar o modelo.
# X_teste e Y_teste serão usados para avaliar o desempenho do modelo.
# test_size=0.3 indica que 30% dos dados serão usados para teste e 70% para treinamento.
# random_state=0 garante que a divisão seja a mesma cada vez que o código for executado, para reprodutibilidade.
X_treinamento, X_teste, Y_treinamento, Y_teste = train_test_split(X, y, test_size=0.3, random_state=0)

In [85]:
# Inicializa o classificador de Árvore de Decisão com parâmetros específicos:
# random_state=1 para reprodutibilidade dos resultados.
# max_depth=8 limita a profundidade máxima da árvore a 8 níveis.
# max_leaf_nodes=6 limita o número máximo de nós folha a 6.
modelo = DecisionTreeClassifier(random_state=1,max_depth=8,max_leaf_nodes=6)
# Treina o modelo usando os dados de treinamento (X_treinamento) e seus rótulos correspondentes (Y_treinamento).
modelo.fit(X_treinamento, Y_treinamento)

DecisionTreeClassifier(max_depth=8, max_leaf_nodes=6, random_state=1)

In [86]:
# Exporta a árvore de decisão treinada para o formato DOT, que pode ser visualizado.
# out_file=None indica que a saída será uma string.
# filled=True preenche os nós com cores para indicar a maioria da classe.
# Obtém os nomes das colunas de 'base' após remover a coluna 'Accident' (índice 7), que é a variável alvo.
feature_names_for_tree = base.drop(base.columns[7], axis=1).columns
dot_data = export_graphviz(modelo, out_file=None, filled=True, feature_names=feature_names_for_tree, class_names=True, rounded =True)
# Cria um objeto Source do graphviz a partir dos dados DOT.
graph = graphviz.Source(dot_data)
# Renderiza a árvore de decisão como um arquivo PNG com o nome "decision_tree".
graph.render("decision_tree",format ="png")
# Abre o arquivo PDF gerado para visualização.
graph.view()

'decision_tree.pdf'

In [87]:
# Usa o modelo treinado para fazer previsões nos dados de teste (X_teste).
# O resultado será um array com as classes previstas para cada amostra no conjunto de teste.
previsoes = modelo.predict(X_teste)

In [88]:
# Exibe as previsões feitas pelo modelo nos dados de teste.
previsoes

array(['Severe', 'None', 'None', ..., 'Moderate', 'None', 'None'],
      dtype=object)

In [89]:
# Calcula a acurácia do modelo: proporção de previsões corretas sobre o total de previsões.
accuracy = accuracy_score(Y_teste, previsoes)

# Calcula a precisão do modelo (weighted average): a capacidade do modelo de não rotular como positiva uma amostra que é negativa.
# average='weighted': calcula a média de cada métrica ponderada pelo suporte (número de instâncias verdadeiras) para cada rótulo.
# zero_division=0: Se não houver amostras para uma classe, a métrica será 0 em vez de levantar um erro.
precision = precision_score(Y_teste, previsoes, average='weighted', zero_division=0)

# Calcula o recall do modelo (weighted average): a capacidade do modelo de encontrar todas as amostras positivas.
recall = recall_score(Y_teste, previsoes, average='weighted', zero_division=0)

# Calcula o F1-score do modelo (weighted average): média harmônica entre precisão e recall.
f1 = f1_score(Y_teste, previsoes, average='weighted', zero_division=0)

# Imprime os valores calculados das métricas de avaliação.
print(f'Acurácia: {accuracy}, Precisão: {precision}, Recall: {recall}, F1-Score: {f1}')

Acurácia: 0.9443333333333334, Precisão: 0.9430691220235771, Recall: 0.9443333333333334, F1-Score: 0.9426693960166915


In [90]:
# Gera um relatório de classificação detalhado, que inclui precisão, recall, f1-score e suporte para cada classe.
# Este relatório é uma ferramenta essencial para avaliar o desempenho do modelo em classificações multi-classe.
report = classification_report(Y_teste, previsoes)
# Imprime o relatório de classificação.
print(report)

              precision    recall  f1-score   support

        Mild       0.91      0.73      0.81       553
    Moderate       0.75      0.73      0.74       473
        None       0.98      1.00      0.99      4294
      Severe       0.88      0.93      0.90       680

    accuracy                           0.94      6000
   macro avg       0.88      0.85      0.86      6000
weighted avg       0.94      0.94      0.94      6000

